In [4]:
!pip install ta

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for ta: filename=ta-0.11.0-py3-none-any.whl size=29422 sha256=1f55745bf19be8d9aa33ca0d1150907537c484e450d4c95d5590dce6b0d2f4b8
  Stored in directory: c:\users\sangam\appdata\local\pip\cache\wheels\a1\d7\29\7781cc5eb9a3659d032d7d15bdd0f49d07d2b24fec29f44bc4
Successfully built ta


In [3]:
import pandas as pd
import numpy as np
import os
import ta

In [5]:
df=pd.read_csv("../data/raw/TSLA.csv")
df.head()

,Date,Close,High,Low,Open,Volume
0,NaN,TSLA,TSLA,TSLA,TSLA,TSLA
1,2015-12-07,15.408666610717773,15.708666801452637,15.076666831970215,15.180000305175781,47163000
2,2015-12-08,15.114666938781738,15.25333309173584,14.946666717529297,15.168000221252441,40314000
3,2015-12-09,14.968000411987305,15.166666984558105,14.714667320251465,15.113332748413086,45867000
4,2015-12-10,15.137999534606934,15.232666969299316,14.909333229064941,14.980667114257812,31075500


In [7]:
df['Date']=pd.to_datetime(df['Date'])
df=df.sort_values('Date')
df.set_index('Date', inplace=True)

In [9]:
# Convert columns to numeric
numeric_cols = ['Open','High','Low','Close','Volume']
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')

#Moving Avgs
df['MA20']=df['Close'].rolling(window=20).mean()
df['MA50']=df['Close'].rolling(window=50).mean()

In [11]:
#RSI
from ta.momentum import RSIIndicator
df['RSI'] = RSIIndicator(df['Close'], window=14).rsi()

In [13]:
#MACD
from ta.trend import MACD

macd=MACD(close=df['Close'])
df['MACD'] = macd.macd()
df['MACD_signal'] = macd.macd_signal()
df['MACD_hist'] = macd.macd_diff()

In [15]:
#Bollinger Bands
from ta.volatility import BollingerBands

indicator_bb = BollingerBands(close=df['Close'], window=20, window_dev=2)

df['BB_high'] = indicator_bb.bollinger_hband()
df['BB_low'] = indicator_bb.bollinger_lband()
df['BB_width'] = df['BB_high'] - df['BB_low']

In [17]:
#Daily Returns
df['Return'] = df['Close'].pct_change(fill_method=None)

In [19]:
#Lag Features
df['Lag1'] = df['Close'].shift(1)
df['Lag2'] = df['Close'].shift(2)
df['Lag5'] = df['Close'].shift(5)

1 = tomorrow's close > today's close
0 = otherwise

In [21]:
#Creating Target Variable
df['Target'] = (df['Close'].shift(-5) > df['Close']).astype(int)

In [34]:
print(df.shape)

(2466, 19)


In [23]:
#Drop NaN values
df.dropna(inplace=True)

In [36]:
print(df.shape)

(2466, 19)


In [25]:
#Inspecting the final dataset
df.head()
df.shape

(2466, 19)

In [27]:
#Save Processed Data
os.makedirs("../data/processed", exist_ok=True)

df.to_csv("../data/processed/TSLA.csv", index=True)

In [29]:
print(f"Saved processed {df}.csv({df.shape})")

Saved processed                  Close        High         Low        Open      Volume  \
Date                                                                     
2016-02-18   11.118000   11.530000   10.984667   11.494667  58314000.0   
2016-02-19   11.105333   11.166000   10.833333   10.910667  44391000.0   
2016-02-22   11.849333   11.927333   11.323333   11.341333  75901500.0   
2016-02-23   11.814000   12.115333   11.578667   11.744000  89766000.0   
2016-02-24   11.933333   11.966667   11.189333   11.516667  80934000.0   
...                ...         ...         ...         ...         ...   
2025-12-01  430.140015  433.660004  425.290009  425.320007  57463600.0   
2025-12-02  429.239990  436.799988  422.119995  430.809998  69336600.0   
2025-12-03  446.739990  447.920013  431.109985  432.100006  87483000.0   
2025-12-04  454.529999  454.630005  445.390015  449.940002  71906500.0   
2025-12-05  455.000000  458.869995  451.660004  452.970001  55325635.0   

                  MA2

In [52]:
import pandas as pd
import os
from ta.momentum import RSIIndicator
from ta.trend import MACD
from ta.volatility import BollingerBands

tickers = ["TSLA", "AMZN", "NVDA", "MSFT", "GOOG"]

for ticker in tickers:
    print(f"\nProcessing {ticker}...")

    # Load data
    df = pd.read_csv(f"../data/raw/{ticker}.csv")
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values('Date')
    df.set_index('Date', inplace=True)

    # Ensure numeric
    numeric_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
    df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')

    # ================= TECHNICAL INDICATORS ================= #

    # Moving averages
    df['MA20'] = df['Close'].rolling(window=20).mean()
    df['MA50'] = df['Close'].rolling(window=50).mean()

    # RSI
    df['RSI'] = RSIIndicator(df['Close'], window=14).rsi()

    # MACD
    macd = MACD(close=df['Close'])
    df['MACD'] = macd.macd()
    df['MACD_signal'] = macd.macd_signal()
    df['MACD_hist'] = macd.macd_diff()

    # Bollinger Bands
    bb = BollingerBands(close=df['Close'], window=20, window_dev=2)
    df['BB_high'] = bb.bollinger_hband()
    df['BB_low'] = bb.bollinger_lband()
    df['BB_width'] = df['BB_high'] - df['BB_low']

    # ================= PRICE-BASED FEATURES ================= #

    # Daily returns
    df['Return'] = df['Close'].pct_change()
    df['Return'] = df['Return'].fillna(0)

    # Momentum (short + medium)
    df['Momentum3'] = df['Close'] / df['Close'].shift(3) - 1
    df['Momentum10'] = df['Close'] / df['Close'].shift(10) - 1

    # Trend regime: MA crossover
    df['Trend'] = (df['MA20'] > df['MA50']).astype(int)

    # Volatility (short)
    df['Volatility10'] = df['Return'].rolling(10).std()

    # Lag features
    df['Lag1'] = df['Close'].shift(1)
    df['Lag5'] = df['Close'].shift(5)

    # ================= TARGET VARIABLE ================= #

    df['Target'] = (df['Close'].shift(-1) > df['Close']).astype(int)

    # ================= CLEAN DATA ================= #

    # Drop rows with NaN after feature creation
    df.dropna(inplace=True)

    # Remove duplicates if any
    df = df.loc[~df.index.duplicated(keep='first')]

    # ================= INSPECT DATA ================= #

    print("Final shape:", df.shape)
    print("Target distribution:")
    print(df['Target'].value_counts(normalize=True))

    # ================= SAVE DATA ================= #

    os.makedirs("../data/processed", exist_ok=True)
    df.to_csv(f"../data/processed/{ticker}.csv", index=True)

    print(f"Saved processed {ticker}.csv")



Processing TSLA...
Final shape: (2466, 22)
Target distribution:
Target
1    0.51987
0    0.48013
Name: proportion, dtype: float64


C:\Users\Sangam\AppData\Local\Temp\ipykernel_11048\745704429.py:46: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df['Return'] = df['Close'].pct_change()
C:\Users\Sangam\AppData\Local\Temp\ipykernel_11048\745704429.py:46: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df['Return'] = df['Close'].pct_change()


Saved processed TSLA.csv

Processing AMZN...
Final shape: (2466, 22)
Target distribution:
Target
1    0.536091
0    0.463909
Name: proportion, dtype: float64
Saved processed AMZN.csv

Processing NVDA...
Final shape: (2466, 22)
Target distribution:
Target
1    0.546229
0    0.453771
Name: proportion, dtype: float64
Saved processed NVDA.csv

Processing MSFT...


C:\Users\Sangam\AppData\Local\Temp\ipykernel_11048\745704429.py:46: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df['Return'] = df['Close'].pct_change()
C:\Users\Sangam\AppData\Local\Temp\ipykernel_11048\745704429.py:46: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df['Return'] = df['Close'].pct_change()


Final shape: (2466, 22)
Target distribution:
Target
1    0.540552
0    0.459448
Name: proportion, dtype: float64
Saved processed MSFT.csv

Processing GOOG...
Final shape: (2466, 22)
Target distribution:
Target
1    0.545418
0    0.454582
Name: proportion, dtype: float64
Saved processed GOOG.csv


C:\Users\Sangam\AppData\Local\Temp\ipykernel_11048\745704429.py:46: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df['Return'] = df['Close'].pct_change()
